# Checkpoint 1 (Week 1): NLP Pipelines with Hugging Face Transformers

This notebook prototypes exactly **two** Hugging Face `pipeline` NLP tasks:
1. **Sentiment analysis** (text-classification)
2. **Named entity recognition** (NER)

It also evaluates and compares **two sentiment models** on a small IMDb subset using `datasets` + `evaluate`, reporting accuracy and average inference latency.

In [ ]:
!pip install -U transformers datasets evaluate accelerate torch torchvision pillow

In [ ]:
import time
import numpy as np
import pandas as pd

from transformers import pipeline
from datasets import load_dataset
import evaluate

import torch


In [ ]:
# Device selection for faster inference when available
DEVICE = 0 if torch.cuda.is_available() else -1
print("DEVICE:", "cuda" if DEVICE == 0 else "cpu")

accuracy_metric = evaluate.load("accuracy")


def avg_latency(predict_fn, inputs, batch_size=16, repeats=1):
    """Average latency (seconds) per example for pipeline inference.

    We time a full pass over `inputs` (batched by `batch_size`).
    """
    n = len(inputs)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        _ = predict_fn(inputs, batch_size=batch_size)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    # total time / n per example
    return (np.mean(times) / n)


def map_label_to_imdb_binary(label):
    """Map pipeline label strings/ids to IMDb binary labels: 1=positive, 0=negative."""
    s = str(label).strip().lower()

    # Direct forms (some models output raw ids like "0"/"1" or words like "positive")
    if s in {"1", "positive"}:
        return 1
    if s in {"0", "negative"}:
        return 0

    # Common patterns across sentiment models
    if "pos" in s:
        return 1
    if "neg" in s:
        return 0

    # Fallback for models that output LABEL_0 / LABEL_1
    if "label_1" in s:
        return 1
    if "label_0" in s:
        return 0

    raise ValueError(f"Unrecognized sentiment label: {label}")


def predict_labels(predict_fn, inputs, batch_size=16):
    """Run pipeline on a list of inputs and return mapped binary labels."""
    outputs = predict_fn(inputs, batch_size=batch_size)
    mapped = [map_label_to_imdb_binary(o["label"]) for o in outputs]
    return mapped


In [ ]:
# Load IMDb dataset subset
# IMDb labels: 0=negative, 1=positive
N = 200  # keep it lightweight for lab/latency timing

imdb = load_dataset("imdb")
texts = imdb["test"]["text"][:N]
true_labels = imdb["test"]["label"][:N]
print("Subset size:", N)
print("Example:", texts[0][:120].replace("\n", " "))
print("True label:", true_labels[0])

In [ ]:
# Two sentiment models to compare (same task)
# Model A: DistilBERT fine-tuned on SST-2
MODEL_A = "distilbert-base-uncased-finetuned-sst-2-english"

# Model B: RoBERTa base fine-tuned for binary sentiment (reviews)
MODEL_B = "AnkitAI/reviews-roberta-base-sentiment-analysis"

# Create pipelines
# 'sentiment-analysis' is an alias for text classification for sentiment models.
sentiment_a = pipeline("sentiment-analysis", model=MODEL_A, device=DEVICE)
sentiment_b = pipeline("sentiment-analysis", model=MODEL_B, device=DEVICE)

# Timing + evaluation settings
BATCH_SIZE = 16


def make_predictor(clf):
    # Pipeline call wrapper so avg_latency can consistently call it
    def _predict(inputs, batch_size=16):
        return clf(
            inputs,
            batch_size=batch_size,
            truncation=True,
            max_length=256,
        )
    return _predict

pred_a = make_predictor(sentiment_a)
pred_b = make_predictor(sentiment_b)

# Evaluate both models on the same IMDb subset

def evaluate_model(pred_fn, model_name):
    pred_labels = predict_labels(pred_fn, texts, batch_size=BATCH_SIZE)

    acc = accuracy_metric.compute(predictions=pred_labels, references=true_labels)["accuracy"]
    latency_s = avg_latency(pred_fn, texts, batch_size=BATCH_SIZE, repeats=1)

    # Failure case collection: show first misclassified example
    failures = []
    for i, (p, t) in enumerate(zip(pred_labels, true_labels)):
        if p != t:
            failures.append((i, texts[i], p, t))
        if len(failures) >= 3:
            break

    return {
        "model": model_name,
        "accuracy": acc,
        "avg_latency_ms": latency_s * 1000.0,
        "first_failures": failures,
    }

res_a = evaluate_model(pred_a, MODEL_A)
res_b = evaluate_model(pred_b, MODEL_B)

print("Model A accuracy:", res_a["accuracy"], "avg latency (ms):", res_a["avg_latency_ms"])
print("Model B accuracy:", res_b["accuracy"], "avg latency (ms):", res_b["avg_latency_ms"])

In [ ]:
# Results table
results = pd.DataFrame([
    {"model": res_a["model"], "accuracy": res_a["accuracy"], "avg_latency_ms": res_a["avg_latency_ms"]},
    {"model": res_b["model"], "accuracy": res_b["accuracy"], "avg_latency_ms": res_b["avg_latency_ms"]},
]).sort_values("accuracy", ascending=False)

results

## Model comparison discussion (Sentiment)

### Why these models?
- `distilbert-base-uncased-finetuned-sst-2-english` (DistilBERT) is a sentiment classifier fine-tuned on **SST-2**. Its model card documents **bias risks** (example: predictions can vary strongly with certain country names even when sentiment should not change) and recommends probing risks on your specific use case.
  - Model card: https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
- `AnkitAI/reviews-roberta-base-sentiment-analysis` (RoBERTa base) is a pretrained binary positive/negative sentiment model fine-tuned on review-style text.
  - Model card/README: https://huggingface.co/AnkitAI/reviews-roberta-base-sentiment-analysis

**Tradeoff expectation:** RoBERTa is typically larger and may be slower than DistilBERT, but often improves accuracy.

### Failure case (concrete)
The code above collects the first few misclassified examples for each model on the IMDb subset. The outputs will appear when you run the notebook.

In [ ]:
# Print one concrete failure case per model

def print_failures(res):
    failures = res["first_failures"]
    if not failures:
        print("No failures found in this subset for:", res["model"])
        return
    print("Failures for:", res["model"])
    for idx, text, pred, true in failures:
        print(f"  Example #{idx}: pred={pred} true={true}")
        print("  Text:", text[:200].replace("\n", " "))
        print()

print_failures(res_a)
print_failures(res_b)


## Task 2 (NER pipeline)

We also implement a second NLP pipeline: **Named Entity Recognition** using a pretrained Transformer model from Hugging Face.

In [ ]:
# NER pipeline (qualitative demo)
# This lab focuses on inference; we don't need full metric evaluation for NER.
NER_MODEL = "dbmdz/bert-large-cased-finetuned-conll03-english"

ner = pipeline("ner", model=NER_MODEL, aggregation_strategy="simple", device=DEVICE)

sample_texts = [
    "Apple is looking at buying U.K. startup for $1 billion.",
    "Barack Obama was the 44th President of the United States.",
]

ner_outputs = ner(sample_texts)
for t, out in zip(sample_texts, ner_outputs):
    print("TEXT:", t)
    print("ENTITIES:")
    for ent in out:
        print("  -", ent)
    print()

## Notes on latency measurement

- Latency is measured by running the sentiment pipeline on the full IMDb subset in batched mode (`batch_size=BATCH_SIZE`).
- The notebook reports **average milliseconds per example**.
- If your runtime varies due to caching/loading, re-run the notebook or increase the repeat count in `avg_latency()`.